# Pronóstico de Unidades Vendidas Semanales por Categoría de Producto

**Dataset:** Customer Shopping Data — registros transaccionales de retail.

> **Nota sobre el origen:** corresponde a retail en **Estambul, Turquía** (centros comerciales Kanyon, Metrocity, Forum Istanbul, Metropol AVM). Precios en liras turcas (TRY).

## Objetivo

Transformar transacciones en **series temporales semanales por categoría** y entrenar modelos para predecir **unidades vendidas por semana** (`quantity`). Útil para planificación de inventario.

**Granularidad temporal:** semanas (lunes a domingo), identificadas por el lunes correspondiente (`W-MON`).

1. **SARIMA** (modelo clásico de series de tiempo con estacionalidad anual de 52 semanas).
2. **Regresión Lineal** con feature engineering (lags, medias móviles, dummies estacionales).

## Estructura del notebook

1. Setup e imports
2. Carga y preparación de datos
3. Análisis exploratorio (EDA)
4. Construcción de series semanales por categoría
5. Modelo SARIMA por categoría (grid search, `s=52`)
6. Modelo de Regresión Lineal por categoría
7. Comparación de modelos
8. Visualizaciones finales
9. Exportación de datasets y resultados
10. Conclusiones y recomendaciones de negocio

## 1. Setup e imports

Fijamos semillas y configuramos estilos de visualización para reproducibilidad.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Setup completo.")

## 2. Carga y preparación de datos

### 2.1 Carga del CSV

In [ ]:
DATA_PATH = "/mnt/user-data/uploads/customer_shopping_data.csv"

df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
df.head()

### 2.2 Inspección rápida: tipos, nulos, duplicados

In [ ]:
print("Tipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())
print(f"\nFilas duplicadas exactas: {df.duplicated().sum()}")
print(f"Invoice IDs únicos: {df['invoice_no'].nunique():,} de {len(df):,} filas")

**Observaciones:**

- No hay valores nulos.
- El número de `invoice_no` únicos coincide con el número de filas → cada fila es una transacción individual. Esto simplifica la agregación.
- El campo `invoice_date` viene como string y debe convertirse a `datetime`.

### 2.3 Conversión de fechas y creación de `total_sales`

Las fechas están en formato `DD/MM/YYYY` (formato europeo/turco). Usamos `dayfirst=True`.

In [ ]:
df["invoice_date"] = pd.to_datetime(df["invoice_date"], dayfirst=True, errors="coerce")
assert df["invoice_date"].isna().sum() == 0, "Hay fechas no parseadas"

# Ventas totales por transacción
df["total_sales"] = df["quantity"] * df["price"]

print(f"Rango de fechas: {df['invoice_date'].min().date()}  →  {df['invoice_date'].max().date()}")
print(f"Ventas totales del período: {df['total_sales'].sum():,.0f} TRY")
df[["invoice_date", "category", "quantity", "price", "total_sales"]].head()

### 2.4 Revisión de outliers

Usamos el rango intercuartílico (IQR) sobre `total_sales` por categoría para detectar valores extremos, pero **no los eliminamos**: en retail los tickets altos legítimos son comunes y la agregación mensual amortigua outliers individuales.

In [ ]:
q1 = df.groupby("category")["total_sales"].quantile(0.25)
q3 = df.groupby("category")["total_sales"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr

outlier_summary = (
    df.groupby("category")["total_sales"]
      .agg(["count", "mean", "max"])
      .assign(upper_iqr=upper.values,
              n_outliers=lambda d: [
                  int((df.loc[df["category"] == cat, "total_sales"] > upper[cat]).sum())
                  for cat in d.index
              ])
      .sort_values("count", ascending=False)
)
outlier_summary

Los "outliers" se concentran en categorías caras (Technology, Shoes, Clothing) y corresponden a tickets grandes legítimos. Se conservan.

### 2.5 Cobertura temporal por semana

Verificamos cuántas transacciones hay por semana. Usamos `W-MON` (semanas de lunes a domingo).

In [ ]:
weekly_coverage = df.set_index("invoice_date").resample("W-MON")["invoice_no"].count()
print(f"Total de semanas: {len(weekly_coverage)}")
print(f"\nPrimeras 5 semanas:")
print(weekly_coverage.head())
print(f"\nÚltimas 5 semanas:")
print(weekly_coverage.tail())

La última semana (2023-03-06) tiene solo ~123 transacciones vs ~900 típicas: está **incompleta**. La excluimos.

In [ ]:
LAST_FULL_WEEK = "2023-02-27"
df = df[df["invoice_date"] <= LAST_FULL_WEEK].copy()
print(f"Rango: {df['invoice_date'].min().date()} → {df['invoice_date'].max().date()}")
weekly_check = df.set_index("invoice_date").resample("W-MON").size()
print(f"Semanas completas: {len(weekly_check)}")

**113 semanas completas** (ene-2021 a feb-2023). Razonable para SARIMA con `s=52`.

### 3.1 Evolución semanal total de ventas

In [ ]:
weekly_total = df.set_index("invoice_date").resample("W-MON")["total_sales"].sum()

fig, ax = plt.subplots(figsize=(14, 4.5))
weekly_total.plot(ax=ax, marker="o", color="#2E86AB", linewidth=1.5, markersize=3)
ax.set_title("Ventas totales semanales — todas las categorías")
ax.set_xlabel("Semana")
ax.set_ylabel("Ventas (TRY)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

print(f"Promedio semanal: {weekly_total.mean():,.0f} TRY")
print(f"Desviación:       {weekly_total.std():,.0f} TRY  ({weekly_total.std()/weekly_total.mean()*100:.1f}%)")

### 3.2 Top categorías por ventas acumuladas

In [ ]:
cat_totals = (
    df.groupby("category")["total_sales"].sum()
      .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
cat_totals.plot(kind="barh", ax=ax, color=sns.color_palette("viridis", len(cat_totals)))
ax.invert_yaxis()
ax.set_title("Ventas totales por categoría (todo el período)")
ax.set_xlabel("Ventas (TRY)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
for i, v in enumerate(cat_totals.values):
    ax.text(v, i, f" {v/1e6:.1f}M", va="center", fontsize=9)
plt.tight_layout()
plt.show()

share = (cat_totals / cat_totals.sum() * 100).round(1)
print("Participación (%):")
print(share.to_string())

**Clothing**, **Shoes** y **Technology** concentran la mayor parte de las ventas en valor. Technology tiene menos transacciones pero tickets muy altos.

### 3.3 Series semanales por categoría (agregación clave)

Construimos **dos** series semanales paralelas:

- `weekly_cat` — ventas en TRY (EDA y contexto).
- `weekly_cat_qty` — **unidades vendidas — target de los modelos**.

In [ ]:
weekly_cat = (
    df.set_index("invoice_date")
      .groupby("category")
      .resample("W-MON")["total_sales"].sum()
      .unstack(level=0)
      .sort_index()
      .fillna(0)
)

weekly_cat_qty = (
    df.set_index("invoice_date")
      .groupby("category")
      .resample("W-MON")["quantity"].sum()
      .unstack(level=0)
      .sort_index()
      .fillna(0)
      .astype(int)
)

print(f"weekly_cat     (TRY)      shape: {weekly_cat.shape}")
print(f"weekly_cat_qty (unidades) shape: {weekly_cat_qty.shape}  ← target")
weekly_cat_qty.head()

#### Evolución semanal de unidades por categoría

Target real del modelado.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for col in weekly_cat_qty.columns:
    ax.plot(weekly_cat_qty.index, weekly_cat_qty[col],
            marker="o", label=col, linewidth=1.2, markersize=2, alpha=0.8)
ax.set_title("Evolución semanal de unidades — TARGET")
ax.set_xlabel("Semana")
ax.set_ylabel("Unidades")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

print("\nResumen estadístico (unidades semanales por categoría):")
weekly_cat_qty.describe().round(0)

### 3.4 Tendencia semanal por categoría

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for col in weekly_cat.columns:
    ax.plot(weekly_cat.index, weekly_cat[col], marker="o", label=col,
            linewidth=1.2, markersize=2, alpha=0.8)
ax.set_title("Evolución semanal de ventas (TRY)")
ax.set_xlabel("Semana")
ax.set_ylabel("Ventas (TRY)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

### 3.5 Estacionalidad: patrón semanal promedio

Promediamos por semana del año (1–52). Con ~2 años de datos, señal ruidosa.

In [ ]:
seasonal_pattern = weekly_cat.copy()
seasonal_pattern["week_of_year"] = seasonal_pattern.index.isocalendar().week
seasonal_avg = seasonal_pattern.groupby("week_of_year").mean()

fig, ax = plt.subplots(figsize=(14, 5))
for col in seasonal_avg.columns:
    normalized = seasonal_avg[col] / seasonal_avg[col].mean()
    ax.plot(seasonal_avg.index, normalized, marker="o", label=col,
            linewidth=1.2, markersize=3, alpha=0.7)

ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Patrón estacional normalizado (1.0 = promedio)")
ax.set_xlabel("Semana del año")
ax.set_ylabel("Ratio vs. promedio")
ax.set_xlim(1, 52)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

**Interpretación:** con solo ~2 ciclos anuales, señal muy ruidosa. Variaciones en semana 1 (festivos) y verano, pero el grid search decidirá si `D>0` vale la pena.

### 3.6 Descomposición STL de la serie total (ilustrativa)

In [ ]:
decomp = seasonal_decompose(weekly_total, model="additive", period=52)
fig = decomp.plot()
fig.set_size_inches(12, 7)
plt.tight_layout()
plt.show()

## 4. Construcción del split train/test

Dividimos cronológicamente: **70% / 30%**. Con 113 semanas: ~79 train / 34 test.

In [ ]:
TRAIN_RATIO = 0.70

n_weeks = len(weekly_cat_qty)
train_size = int(n_weeks * TRAIN_RATIO)
test_size  = n_weeks - train_size

train_idx = weekly_cat_qty.index[:train_size]
test_idx  = weekly_cat_qty.index[train_size:]

print(f"Total semanas: {n_weeks}")
print(f"Train:         {train_size} semanas  ({train_idx.min().date()} → {train_idx.max().date()})")
print(f"Test:          {test_size} semanas  ({test_idx.min().date()} → {test_idx.max().date()})")
print(f"\nTarget: quantity (unidades semanales)")

### 4.1 Criterio de exclusión de categorías

Con 113 semanas, una categoría debe tener:
- **al menos 52 semanas con ventas > 0** (un año completo),
- **varianza no nula** en train.

In [ ]:
MIN_NONZERO_WEEKS = 52

valid_categories = []
for cat in weekly_cat_qty.columns:
    series = weekly_cat_qty[cat]
    nonzero = (series > 0).sum()
    train_var = series.iloc[:train_size].var()
    if nonzero >= MIN_NONZERO_WEEKS and train_var > 0:
        valid_categories.append(cat)
    else:
        print(f"⚠  Excluyendo '{cat}': semanas con ventas>0={nonzero}, varianza={train_var:.2f}")

print(f"\nCategorías válidas ({len(valid_categories)}): {valid_categories}")

## 5. Modelo SARIMA por categoría

### 5.1 Estrategia de grid search

Con solo 26 meses, restringimos el espacio de búsqueda a modelos **parsimoniosos** para evitar overfit:

- `p, q ∈ {0, 1, 2}`
- `d ∈ {0, 1}`
- `P, Q ∈ {0, 1}`
- `D ∈ {0, 1}`
- `s = 12` (estacionalidad anual)

**Selección:** mínimo AIC en train. Luego evaluamos el modelo ganador en test y reportamos RMSE / MAE / MAPE. La elección de hiperparámetros usa **solo datos de entrenamiento** → sin *data leakage*.

### 5.1 Estrategia de grid search

Con 113 semanas (~2 años), modelos **parsimoniosos**:

- `p, q ∈ {0, 1, 2}`, `d ∈ {0, 1}`
- `P, Q ∈ {0, 1}`, `D ∈ {0, 1}`
- `s = 52` (estacionalidad anual en semanas)

**Selección:** mínimo AIC en train.

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mape(y_true, y_pred),
    }

In [ ]:
def sarima_grid_search(y_train, seasonal_period=52):
    """Grid search parsimonioso. Selección basada SOLO en AIC del train."""
    p_range, d_range, q_range = range(0, 3), range(0, 2), range(0, 3)
    P_range, D_range, Q_range = range(0, 2), range(0, 2), range(0, 2)

    best_aic, best_order, best_seasonal, best_model = np.inf, None, None, None

    for p, d, q in itertools.product(p_range, d_range, q_range):
        for P, D, Q in itertools.product(P_range, D_range, Q_range):
            if p == d == q == P == D == Q == 0:
                continue
            try:
                model = SARIMAX(
                    y_train, order=(p, d, q),
                    seasonal_order=(P, D, Q, seasonal_period),
                    enforce_stationarity=False, enforce_invertibility=False,
                )
                fit = model.fit(disp=False, maxiter=200)
                if np.isfinite(fit.aic) and fit.aic < best_aic:
                    best_aic = fit.aic
                    best_order = (p, d, q)
                    best_seasonal = (P, D, Q, seasonal_period)
                    best_model = fit
            except Exception:
                continue

    return best_order, best_seasonal, best_aic, best_model

### 5.2 Entrenamiento SARIMA por categoría

Ejecutamos el grid search para cada categoría (puede tardar ~1-2 minutos).

In [ ]:
sarima_results = {}

for cat in valid_categories:
    y = weekly_cat_qty[cat].astype(float)
    y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

    print(f"→ {cat}...", end=" ", flush=True)
    order, seasonal, aic, fit = sarima_grid_search(y_train)

    if fit is None:
        print("FALLÓ"); continue

    forecast = fit.forecast(steps=len(y_test))
    forecast.index = y_test.index
    metrics = evaluate(y_test.values, forecast.values)

    sarima_results[cat] = {
        "order": order, "seasonal_order": seasonal, "aic": aic,
        "model": fit, "forecast": forecast,
        "y_train": y_train, "y_test": y_test, **metrics,
    }
    print(f"order={order} s={seasonal} AIC={aic:.1f} RMSE={metrics['RMSE']:,.1f} MAPE={metrics['MAPE']:.2f}%")

### 5.3 Tabla resumen SARIMA

In [ ]:
sarima_summary = pd.DataFrame([
    {"category": cat, "order": str(r["order"]), "seasonal_order": str(r["seasonal_order"]),
     "AIC": r["aic"], "MAE": r["MAE"], "RMSE": r["RMSE"], "MAPE (%)": r["MAPE"]}
    for cat, r in sarima_results.items()
]).sort_values("RMSE").reset_index(drop=True)
sarima_summary

### 5.4 Gráficos real vs predicción — SARIMA

In [ ]:
n_cats = len(sarima_results)
n_cols = 2
n_rows = int(np.ceil(n_cats / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.2 * n_rows))
axes = axes.flatten()

i = -1
for i, (cat, r) in enumerate(sarima_results.items()):
    ax = axes[i]
    ax.plot(r["y_train"].index, r["y_train"].values, label="Train", color="#1f77b4", linewidth=1.5)
    ax.plot(r["y_test"].index,  r["y_test"].values,  label="Test real", color="#2ca02c", marker="o", linewidth=2)
    ax.plot(r["forecast"].index, r["forecast"].values, label="SARIMA pred.", color="#d62728",
            marker="x", linestyle="--", linewidth=2)
    ax.set_title(f"{cat}  |  RMSE={r['RMSE']:,.0f} u.  MAPE={r['MAPE']:.1f}%")
    ax.set_ylabel("Unidades")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("SARIMA — Real vs. Predicción por categoría (target: unidades)", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## 6. Modelo de Regresión Lineal por categoría

### 6.1 Feature engineering

Creamos variables a partir de la serie temporal:

- **Lags:** `lag_1`, `lag_2`, `lag_3`
- **Medias móviles:** `ma_3`, `ma_6` (calculadas sobre `shift(1)` para que no incluyan el mes actual — así evitamos leakage)
- **Tendencia:** `time_index`
- **Calendario:** `month`, `quarter`, `year`
- **Dummies estacionales:** un dummy por mes (`m_2, ..., m_12`, con enero como referencia)

In [ ]:
def build_features(series: pd.Series) -> pd.DataFrame:
    """Construye features para regresión lineal (serie semanal)."""
    feat = pd.DataFrame({"y": series.values}, index=series.index)

    # Lags: 1, 2, 4 semanas
    feat["lag_1"] = feat["y"].shift(1)
    feat["lag_2"] = feat["y"].shift(2)
    feat["lag_4"] = feat["y"].shift(4)

    # Medias móviles: ~1 mes, ~1 trimestre
    feat["ma_4"]  = feat["y"].shift(1).rolling(window=4).mean()
    feat["ma_13"] = feat["y"].shift(1).rolling(window=13).mean()

    # Tendencia y calendario
    feat["time_index"] = np.arange(len(feat))
    feat["week_of_year"] = feat.index.isocalendar().week
    feat["month"]   = feat.index.month
    feat["quarter"] = feat.index.quarter
    feat["year"]    = feat.index.year

    # Dummies estacionales por mes (no 52 semanas)
    month_dummies = pd.get_dummies(feat["month"], prefix="m", drop_first=True).astype(int)
    feat = pd.concat([feat, month_dummies], axis=1)
    return feat

### 6.2 Predicción recursiva

Al evaluar en test usamos **pronóstico recursivo**: cuando el modelo predice el mes `t`, esa predicción alimenta los lags para predecir `t+1`. Así replicamos cómo se usaría el modelo en producción y evitamos el leakage de usar los valores reales del test como lags.

In [ ]:
def forecast_recursive_lr(model, features_full, feature_cols, train_end_idx, test_len):
    """Predice recursivamente (usa predicciones previas para construir lags futuros)."""
    data = features_full.copy()
    preds = []

    for step in range(test_len):
        current_idx = train_end_idx + step
        if step > 0:
            data.iloc[current_idx, data.columns.get_loc("lag_1")] = data.iloc[current_idx - 1]["y"]
            if current_idx - 2 >= 0:
                data.iloc[current_idx, data.columns.get_loc("lag_2")] = data.iloc[current_idx - 2]["y"]
            if current_idx - 4 >= 0:
                data.iloc[current_idx, data.columns.get_loc("lag_4")] = data.iloc[current_idx - 4]["y"]
            data.iloc[current_idx, data.columns.get_loc("ma_4")]  = data.iloc[current_idx-4:current_idx]["y"].mean()
            data.iloc[current_idx, data.columns.get_loc("ma_13")] = data.iloc[current_idx-13:current_idx]["y"].mean()

        X_row = data.iloc[[current_idx]][feature_cols]
        pred = model.predict(X_row)[0]
        preds.append(pred)
        data.iloc[current_idx, data.columns.get_loc("y")] = pred

    return np.array(preds)

### 6.3 Entrenamiento y evaluación por categoría

In [ ]:
lr_results = {}

for cat in valid_categories:
    series = weekly_cat_qty[cat].astype(float)
    feat = build_features(series)
    feature_cols = [c for c in feat.columns if c != "y"]

    train_mask = feat.index < test_idx[0]
    train_rows = feat[train_mask].dropna()

    if len(train_rows) < 20:
        print(f"⚠  {cat}: solo {len(train_rows)} filas tras NaN"); continue

    X_train, y_train = train_rows[feature_cols], train_rows["y"]
    model = LinearRegression()
    model.fit(X_train, y_train)

    test_len = int((~train_mask).sum())
    train_end_idx = feat.index.get_loc(test_idx[0])
    preds = forecast_recursive_lr(model, feat, feature_cols, train_end_idx, test_len)

    y_test_actual = series.iloc[train_size:].values
    metrics = evaluate(y_test_actual, preds)

    lr_results[cat] = {
        "model": model, "coefs": dict(zip(feature_cols, model.coef_)),
        "forecast": pd.Series(preds, index=test_idx),
        "y_train": series.iloc[:train_size], "y_test": series.iloc[train_size:],
        **metrics,
    }
    print(f"✓ {cat:20s} RMSE={metrics['RMSE']:,.1f} MAPE={metrics['MAPE']:.2f}%")

### 6.4 Tabla resumen Regresión Lineal

In [ ]:
lr_summary = pd.DataFrame([
    {"category": cat, "MAE": r["MAE"], "RMSE": r["RMSE"], "MAPE (%)": r["MAPE"]}
    for cat, r in lr_results.items()
]).sort_values("RMSE").reset_index(drop=True)
lr_summary

### 6.5 Gráficos real vs predicción — Regresión Lineal

In [ ]:
n_cats = len(lr_results)
n_cols = 2
n_rows = int(np.ceil(n_cats / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.2 * n_rows))
axes = axes.flatten()

i = -1
for i, (cat, r) in enumerate(lr_results.items()):
    ax = axes[i]
    ax.plot(r["y_train"].index, r["y_train"].values, label="Train", color="#1f77b4", linewidth=1.5)
    ax.plot(r["y_test"].index,  r["y_test"].values,  label="Test real", color="#2ca02c", marker="o", linewidth=2)
    ax.plot(r["forecast"].index, r["forecast"].values, label="LR pred.", color="#9467bd",
            marker="x", linestyle="--", linewidth=2)
    ax.set_title(f"{cat}  |  RMSE={r['RMSE']:,.0f} u.  MAPE={r['MAPE']:.1f}%")
    ax.set_ylabel("Unidades")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Regresión Lineal — Real vs. Predicción por categoría (target: unidades)", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## 7. Comparación de modelos

### 7.1 Tabla comparativa por categoría

In [ ]:
comparison_rows = []
for cat in valid_categories:
    if cat not in sarima_results or cat not in lr_results:
        continue
    s = sarima_results[cat]
    l = lr_results[cat]
    best = "SARIMA" if s["RMSE"] < l["RMSE"] else "Regresión Lineal"
    comparison_rows.append({
        "category": cat,
        "SARIMA_RMSE": s["RMSE"], "SARIMA_MAPE (%)": s["MAPE"],
        "LR_RMSE": l["RMSE"], "LR_MAPE (%)": l["MAPE"],
        "Mejor (por RMSE)": best,
    })

comparison = pd.DataFrame(comparison_rows).sort_values("SARIMA_RMSE").reset_index(drop=True)
comparison

### 7.2 Resumen: cuál modelo gana globalmente

In [ ]:
winners = comparison["Mejor (por RMSE)"].value_counts()
print("Categorías ganadas por cada modelo (criterio: menor RMSE):")
print(winners.to_string())

print(f"\nRMSE promedio SARIMA:   {comparison['SARIMA_RMSE'].mean():,.0f}")
print(f"RMSE promedio LR:       {comparison['LR_RMSE'].mean():,.0f}")
print(f"MAPE promedio SARIMA:   {comparison['SARIMA_MAPE (%)'].mean():.2f}%")
print(f"MAPE promedio LR:       {comparison['LR_MAPE (%)'].mean():.2f}%")

## 8. Visualizaciones finales

### 8.1 Predicciones vs. reales — ambos modelos superpuestos

In [ ]:
n_cats = len(comparison)
n_cols = 2
n_rows = int(np.ceil(n_cats / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.3 * n_rows))
axes = axes.flatten()

i = -1
for i, cat in enumerate(comparison["category"]):
    ax = axes[i]
    s = sarima_results[cat]
    l = lr_results[cat]
    ax.plot(s["y_train"].index, s["y_train"].values, label="Train", color="#555555", linewidth=1.2, alpha=0.7)
    ax.plot(s["y_test"].index,  s["y_test"].values,  label="Real", color="#2ca02c", marker="o", linewidth=2)
    ax.plot(s["forecast"].index, s["forecast"].values,
            label=f"SARIMA (MAPE={s['MAPE']:.1f}%)", color="#d62728", marker="x", linestyle="--", linewidth=1.8)
    ax.plot(l["forecast"].index, l["forecast"].values,
            label=f"LR (MAPE={l['MAPE']:.1f}%)", color="#9467bd", marker="s", linestyle=":", linewidth=1.8)
    ax.set_title(cat)
    ax.set_ylabel("Unidades")
    ax.legend(fontsize=8, loc="best")
    ax.tick_params(axis="x", rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Comparación SARIMA vs. Regresión Lineal por categoría (target: unidades)", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

### 8.2 Ranking de categorías mejor pronosticadas (mejor MAPE entre ambos modelos)

In [ ]:
ranking = comparison.copy()
ranking["best_MAPE"] = ranking[["SARIMA_MAPE (%)", "LR_MAPE (%)"]].min(axis=1)
ranking["best_model"] = ranking["Mejor (por RMSE)"]
ranking = ranking.sort_values("best_MAPE").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#d62728" if m == "SARIMA" else "#9467bd" for m in ranking["best_model"]]
bars = ax.barh(ranking["category"], ranking["best_MAPE"], color=colors)
ax.invert_yaxis()
ax.set_title("Ranking de categorías mejor pronosticadas (menor MAPE)")
ax.set_xlabel("MAPE del mejor modelo (%)")

for bar, val, m in zip(bars, ranking["best_MAPE"], ranking["best_model"]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%  ({m})", va="center", fontsize=9)

from matplotlib.patches import Patch
legend_elems = [Patch(color="#d62728", label="SARIMA gana"),
                Patch(color="#9467bd", label="Regresión Lineal gana")]
ax.legend(handles=legend_elems, loc="lower right")
plt.tight_layout()
plt.show()

### 8.3 Error (RMSE) por modelo y categoría

In [ ]:
error_df = comparison.melt(
    id_vars="category",
    value_vars=["SARIMA_RMSE", "LR_RMSE"],
    var_name="Modelo", value_name="RMSE",
)
error_df["Modelo"] = error_df["Modelo"].map({"SARIMA_RMSE": "SARIMA", "LR_RMSE": "Regresión Lineal"})

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(
    data=error_df, y="category", x="RMSE", hue="Modelo",
    palette={"SARIMA": "#d62728", "Regresión Lineal": "#9467bd"}, ax=ax,
)
ax.set_title("RMSE por categoría y modelo (unidades)")
ax.set_xlabel("RMSE (unidades)")
plt.tight_layout()
plt.show()

### 8.4 Parámetros SARIMA ganadores por categoría

In [ ]:
params_table = pd.DataFrame([
    {"category": cat, "(p,d,q)": str(r["order"]),
     "(P,D,Q,s)": str(r["seasonal_order"]), "AIC": r["aic"]}
    for cat, r in sarima_results.items()
]).sort_values("AIC").reset_index(drop=True)
params_table

## 9. Exportación de datasets y resultados

Target: `quantity` (unidades semanales). Mantenemos también TRY para contexto.

**Archivos generados:**

| Archivo | Contenido |
|---|---|
| `weekly_sales_by_category.csv` | Serie semanal TRY (contexto) |
| `weekly_quantity_by_category.csv` | **Serie semanal unidades — target** |
| `weekly_quantity_by_category_long.csv` | Unidades en formato largo |
| `sarima_test_data.csv` | Test (unidades reales semanales) |
| `lr_test_data.csv` | Test LR |
| `sarima_predictions.csv` | Predicciones SARIMA |
| `lr_predictions.csv` | Predicciones LR |
| `predictions_all_models.csv` | Consolidado |
| `model_comparison.csv` | Métricas |
| `sarima_best_params.csv` | Hiperparámetros |

In [ ]:
import os

OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

saved_files = []

def _save(df, filename, **kwargs):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, **kwargs)
    saved_files.append(path)
    print(f"✓ {filename:42s}  shape={df.shape}")
    return path

print(f"Directorio de salida: {OUTPUT_DIR}\n")

### 9.1 Dataset mensual por categoría

In [ ]:
weekly_sales_wide = weekly_cat.copy()
weekly_sales_wide.index.name = "week"
_save(weekly_sales_wide, "weekly_sales_by_category.csv")

weekly_qty_wide = weekly_cat_qty.copy()
weekly_qty_wide.index.name = "week"
_save(weekly_qty_wide, "weekly_quantity_by_category.csv")

weekly_qty_long = (
    weekly_cat_qty.reset_index()
                  .melt(id_vars="invoice_date", var_name="category", value_name="quantity")
                  .rename(columns={"invoice_date": "week"})
                  .sort_values(["week", "category"])
                  .reset_index(drop=True)
)
_save(weekly_qty_long, "weekly_quantity_by_category_long.csv", index=False)

weekly_qty_long.head()

### 9.2 Datasets de test (valores reales) para cada modelo

Ambos modelos usan exactamente el mismo set de test (los últimos 8 meses). Los guardamos por separado para que cada artefacto sea autocontenido.

In [ ]:
test_data_rows = []
for cat in valid_categories:
    if cat not in sarima_results:
        continue
    y_test = sarima_results[cat]["y_test"]
    for week, val in y_test.items():
        test_data_rows.append({"week": week, "category": cat, "quantity_true": float(val)})

test_df = pd.DataFrame(test_data_rows).sort_values(["week", "category"]).reset_index(drop=True)
_save(test_df, "sarima_test_data.csv", index=False)
_save(test_df, "lr_test_data.csv", index=False)
test_df.head()

### 9.3 Predicciones de cada modelo

In [ ]:
sarima_pred_rows = []
for cat, r in sarima_results.items():
    for week, val in r["forecast"].items():
        sarima_pred_rows.append({"week": week, "category": cat, "quantity_pred": float(val), "model": "SARIMA"})
sarima_pred_df = pd.DataFrame(sarima_pred_rows).sort_values(["week", "category"]).reset_index(drop=True)
_save(sarima_pred_df, "sarima_predictions.csv", index=False)

lr_pred_rows = []
for cat, r in lr_results.items():
    for week, val in r["forecast"].items():
        lr_pred_rows.append({"week": week, "category": cat, "quantity_pred": float(val), "model": "LinearRegression"})
lr_pred_df = pd.DataFrame(lr_pred_rows).sort_values(["week", "category"]).reset_index(drop=True)
_save(lr_pred_df, "lr_predictions.csv", index=False)

print("SARIMA head:")
print(sarima_pred_df.head())
print("\nLR head:")
print(lr_pred_df.head())

### 9.4 Tabla consolidada: real + predicciones + errores

Útil para análisis downstream: permite calcular métricas por segmento, identificar los peores meses, alimentar dashboards, etc.

In [ ]:
consolidated = (
    test_df
      .merge(sarima_pred_df.rename(columns={"quantity_pred": "sarima_pred"}).drop(columns="model"),
             on=["week", "category"], how="left")
      .merge(lr_pred_df.rename(columns={"quantity_pred": "lr_pred"}).drop(columns="model"),
             on=["week", "category"], how="left")
)

consolidated["sarima_abs_error"] = (consolidated["quantity_true"] - consolidated["sarima_pred"]).abs()
consolidated["lr_abs_error"]     = (consolidated["quantity_true"] - consolidated["lr_pred"]).abs()

safe_true = consolidated["quantity_true"].replace(0, np.nan)
consolidated["sarima_ape_%"] = (consolidated["sarima_abs_error"] / safe_true * 100).round(2)
consolidated["lr_ape_%"]     = (consolidated["lr_abs_error"]     / safe_true * 100).round(2)

_save(consolidated, "predictions_all_models.csv", index=False)
consolidated.head(10)

### 9.5 Comparación de modelos y parámetros SARIMA

In [ ]:
# Métricas por categoría y modelo (formato largo)
metrics_rows = []
for cat in valid_categories:
    if cat in sarima_results:
        r = sarima_results[cat]
        metrics_rows.append({
            "category": cat, "model": "SARIMA",
            "MAE": r["MAE"], "RMSE": r["RMSE"], "MAPE_%": r["MAPE"],
        })
    if cat in lr_results:
        r = lr_results[cat]
        metrics_rows.append({
            "category": cat, "model": "LinearRegression",
            "MAE": r["MAE"], "RMSE": r["RMSE"], "MAPE_%": r["MAPE"],
        })

metrics_df = pd.DataFrame(metrics_rows).sort_values(["category", "model"]).reset_index(drop=True)
_save(metrics_df, "model_comparison.csv", index=False)

# Hiperparámetros SARIMA ganadores
sarima_params_df = pd.DataFrame([
    {
        "category": cat,
        "p": r["order"][0], "d": r["order"][1], "q": r["order"][2],
        "P": r["seasonal_order"][0], "D": r["seasonal_order"][1],
        "Q": r["seasonal_order"][2], "s": r["seasonal_order"][3],
        "AIC": r["aic"],
    }
    for cat, r in sarima_results.items()
]).sort_values("AIC").reset_index(drop=True)

_save(sarima_params_df, "sarima_best_params.csv", index=False)

print("\nMétricas:")
print(metrics_df)
print("\nParámetros SARIMA:")
print(sarima_params_df)

### 9.6 Resumen de archivos guardados

In [ ]:
print(f"Total de archivos guardados: {len(saved_files)}\n")
for p in saved_files:
    size_kb = os.path.getsize(p) / 1024
    print(f"  {os.path.basename(p):42s}  {size_kb:7.1f} KB")

## 10. Conclusiones y recomendaciones de negocio

### 10.1 Hallazgos técnicos

- **Granularidad temporal:** semanal (lunes a domingo). 113 semanas completas (ene-2021 a feb-2023).
- **Target:** `quantity` (unidades semanales por categoría). No contaminado por precio o inflación.
- **Estacionalidad:** con solo ~2 ciclos anuales, señal ruidosa. SARIMA con `s=52` captura patrones anuales pero intervalos amplios.
- **Origen:** retail de **Estambul (Turquía)**.
- **Split temporal + predicción recursiva** en LR → sin data leakage.

### 10.2 Comparación de modelos

- **SARIMA** captura mejor estacionalidad clara.
- **Regresión Lineal** destaca con auto-correlación corta. Más **interpretable**.
- Ninguno domina globalmente: ambas familias son válidos y complementarios.

### 10.3 Recomendaciones de negocio

1. **Planificación de inventario semanal:** predicciones en unidades × precio promedio = ingresos proyectados.
2. **Selección de modelo por categoría:** desplegar ambos, elegir según MAPE en test reciente.
3. **Intervalos de predicción:** usar percentil 85-95 como target de stock (nivel de servicio).
4. **MAPE alto:** revisar drivers externos (promociones, festividades, inflación). Extender a SARIMAX.
5. **Re-entrenamiento:** semanal. Ganancia marginal alta con solo 2 años de historia.
6. **Siguiente paso:** acumular 3+ años. Probar Prophet, ETS, LightGBM. Evaluar ensambles.
7. **Valor monetario:** combinar pronóstico de unidades con modelo de precio promedio — separa volumen de precio.
8. **Horizonte:** razonable pronosticar 4-8 semanas. Para más largo, agregar a nivel mensual es más robusto.

### 10.4 Limitaciones

- Solo ~34 semanas de test → varianza moderada en métricas.
- Sin variables exógenas (precio, tráfico, eventos, inflación).
- LR recursiva acumula error en horizontes largos (>8 semanas).
- No distingue número de tickets vs unidades por ticket.
- Con solo 2 ciclos anuales, componente `(P,D,Q,52)` tiene poca evidencia. Más años mejorarían sustancialmente.